# Data management recommendations for seismology workflows

This notebook gives some practical recommendations for organizing continuous and event waveform data, and then explains how to access and use **SEISAN** on `newton`.

The main goal is to keep your archive:

- easy to search,
- easy to process with Python,
- compatible with multiple legacy and modern tools,
- reproducible over many years of research.

## 1. Big picture

A useful pattern is to treat **continuous waveform data** and **event waveform data** as two related but distinct products.

### Continuous waveform data
Continuous data are best treated as a long-term archive. They should be stored in a standardized layout that is easy to read by many tools.

### Event waveform data
Event data are best treated as extracted products associated with individual detections or cataloged events. These are often easier to review, pick, classify, and relocate when grouped event-by-event.

This leads naturally to using:

- **SDS archives** for continuous data, and
- **SEISAN-style event files** for event waveform data.

## 2. Recommendation for continuous data

### Store continuous data as:

- **one MiniSEED file per day per SEED id**
- organized in **SeisComP Data Structure (SDS)** layout

This is a very good default for long-term continuous archives because it is:

- simple,
- standard,
- compatible with ObsPy and many other tools,
- efficient for day-based processing,
- well suited to RSAM, detection, and archive conversion workflows.

A typical SDS path looks like:

```text
SDS_ROOT/YYYY/NET/STA/CHAN.D/NET.STA.LOC.CHAN.D.YYYY.JJJ
```

where:

- `YYYY` = 4-digit year
- `NET` = network code
- `STA` = station code
- `LOC` = location code
- `CHAN` = channel code
- `JJJ` = Julian day

### Why this is recommended

A one-file-per-day-per-SEED-id structure keeps the archive modular and easy to repair. If one day is corrupt or missing, you only lose one day for one channel, not an entire month or station.

It also matches the natural rhythm of many continuous-data workflows:

- reading one day at a time,
- computing daily metrics,
- running detectors day-by-day,
- parallelizing work across days or channels.

## 3. SDS and Antelope can work together

An SDS archive does **not** prevent you from using Antelope.

A useful strategy is:

1. archive the continuous data in **MiniSEED in SDS layout**,
2. then build an **Antelope `wfdisc` table** that points to those MiniSEED files.

This gives you the best of both worlds:

- **SDS** as a clean archival structure,
- **Antelope** as a database/query layer.

So SDS and Antelope are not competing ideas. They can work very well together.

### Practical consequence

You do **not** need to duplicate the waveform data just to use Antelope. In many cases, it is enough to maintain:

- one authoritative SDS archive, and
- a `wfdisc` table referring to those files.

## 4. Recommendation for event waveform data: **Seisan** file-naming convention

### 4.1 Store event waveform data as:

- **one MiniSEED file per event**
- **not** one file per SEED id
- named using **SEISAN event waveform naming conventions**
- stored in a directory like:

```text
WAV/{DBNAME}/{YYYY}/{MM}
```

where:

- `DBNAME` is the database name, up to **5 characters**,
- `YYYY` is the 4-digit year,
- `MM` is the 2-digit month.

This event-per-file approach is very practical for event review and catalog work. It groups the channels belonging to the same event into one waveform file, which is exactly how many SEISAN workflows expect to see event data.


### 4.2 Event Waveform File Naming (SEISAN-style with MiniSEED)

In SEISAN, event waveform files follow a standardized, time-based naming convention that encodes key metadata directly in the filename. This convention is extremely useful for organizing large event datasets and linking waveform data to event metadata stored in REA/.

Even when we use MiniSEED instead of native SEISAN formats, it is strongly recommended to retain this naming convention, with a small modification described below.

⸻

### 4.3 General filename structure

YYYY-MM-DD-HHMM-SSX.DBNAME__NNN

where:
	•	YYYY-MM-DD-HHMM-SS → event origin time
	•	X → file type indicator
	•	DBNAME → database name (padded to 5 characters)
	•	NNN → sequence number (3 digits)

⸻

🔍 Example (MiniSEED version)

2000-01-12-1208-16M.MVO___019


⸻

### 4.4 Field-by-field explanation

1. Origin time

2000-01-12-1208-16

	•	Year, month, day
	•	Hour and minute (HHMM)
	•	Seconds (SS)

👉 This uniquely identifies the event in time.


2. File type indicator

M

In traditional SEISAN:
	•	S → SEISAN waveform format

In our workflow:
	•	M → MiniSEED waveform file

👉 This makes it immediately clear that the file contains MiniSEED data rather than SEISAN binary format.


3. Database name

MVO___

	•	Up to 5 characters
	•	Padded with underscores _ if shorter

Examples:
	•	MVO__
	•	ASNE_
	•	TEST_

4. Sequence number

019

	•	A 3-digit integer
	•	Used if multiple waveform files exist for the same event

📦 What’s inside the file?

Unlike SDS archives (which store one file per station per day), these files contain:

👉 All waveform data for a single event
	•	multiple stations
	•	multiple channels (Z, N, E, etc.)
	•	time window around the event

This makes them ideal for:
	•	event-based analysis
	•	manual picking (SEISAN, Snuffler, etc.)
	•	machine learning datasets


### 4.5 Relationship to REA/ directory

Each waveform file is typically referenced inside a corresponding S-file in:

REA/{DBNAME}/{YYYY}/{MM}

Example entry inside an S-file:

WAV 2000-01-12-1208-16M.MVO___019

👉 This provides a direct link between:
	•	event metadata (picks, location, classification)
	•	waveform data


### 4.6 Comparison with SDS naming

System	File type	Organization	Naming style
SEISAN WAV	Event-based	one file per event	time-based
SDS	Continuous	one file per station/day	network/station/channel

👉 These systems complement each other:
	•	SDS → continuous monitoring, processing
	•	SEISAN-style WAV → event analysis and interpretation


### 4.7 Recommendation

For this course and your projects:
	•	Use MiniSEED for waveform storage
	•	Use SEISAN-style event filenames with M instead of S
	•	Store files in:

WAV/{DBNAME}/{YYYY}/{MM}

This ensures compatibility with:
	•	SEISAN tools
	•	ObsPy workflows
	•	Antelope (via wfdisc)
	•	your own analysis pipelines


### 4.8 Key takeaway

The filename is not just a label — it is a compact, structured summary of the event.

Using this convention will give you the option of using Seisan to process/analyze your event catalogs.


## 5. Why SEISAN-style event files are still useful

SEISAN is perhaps the most comprehensive traditional system for:

- event browsing,
- phase picking,
- catalog maintenance,
- magnitude work,
- relocation,
- waveform review tied closely to event metadata.

If you store event waveforms in a SEISAN-compatible `WAV` directory tree, you keep the option to use:

- `EEV`
- `MULPLT`
- `SELECT`
- `COLLECT`
- `AUTOREG`
- and other catalog tools

with much less friction.

## 6. Parallel `WAV` and `REA` directory structure

A very useful SEISAN pattern is to keep event waveform files and event metadata in parallel directory trees.

### Event waveform files
```text
WAV/{DBNAME}/{YYYY}/{MM}
```

### Event metadata / readings / results
```text
REA/{DBNAME}/{YYYY}/{MM}
```

In SEISAN terms, the `REA` directory stores the event metadata in **S-files** in Nordic format.
These S-files contain the **readings** or **results** of analysis, such as:

- origin time,
- hypocenter,
- phase picks,
- amplitudes,
- magnitudes,
- classifications,
- analyst comments,
- pointers to waveform files.

So a sensible mental model is:

- `WAV` = waveform files
- `REA` = readings/results for those waveform files

This separation is very powerful because it keeps raw-ish event waveform data distinct from interpreted event metadata.

## 7. Suggested directory examples

### Continuous archive
```text
/home/glenn/mydata/SDS/2003/MV/MBGA/HHZ.D/MV.MBGA..HHZ.D.2003.182
/home/glenn/mydata/SDS/2003/MV/MBGA/HHN.D/MV.MBGA..HHN.D.2003.182
/home/glenn/mydata/SDS/2003/MV/MBGA/HHE.D/MV.MBGA..HHE.D.2003.182
```

### Event waveform archive
```text
/home/glenn/mydata/seisan/WAV/MVOE_/2003/07/2003-07-01-1234-56S.MVOE__017
```

### Event metadata archive
```text
/home/glenn/mydata/seisan/REA/MVOE_/2003/07/01-1234-56L.S200307
```

The exact event file names can vary depending on your workflow and SEISAN version, but the key idea is the same: **event waveform data in `WAV`, event metadata in `REA`, organized by database/year/month**.

## 8. Practical recommendations for students

### Recommendation 1
Keep one **authoritative archive** for continuous data.

That should usually be your SDS MiniSEED archive.

### Recommendation 2
Treat extracted event waveform files as **derived products**.

These can be regenerated from the continuous archive if needed, provided the event timing and extraction rules are documented.

### Recommendation 3
Do not invent a custom folder structure unless there is a very strong reason.

Use established conventions where possible:

- SDS for continuous data
- SEISAN `WAV/REA` for event data and metadata

### Recommendation 4
Use naming conventions consistently.

Small inconsistencies in network, station, location, or channel naming become major problems later.

### Recommendation 5
Keep metadata as close to the waveform data as possible.

At minimum, preserve:

- station metadata,
- instrument responses,
- event IDs,
- extraction times,
- provenance of picks and locations.

### Recommendation 6
Prefer open and well-supported formats.

- MiniSEED for waveforms
- StationXML for station metadata
- Nordic or QuakeML for event metadata, depending on workflow

## 9. How to use SEISAN on `newton`

This section explains a practical workflow for accessing SEISAN remotely from your own laptop or desktop.

### 9.1 Step 1: set up SSH keys first

Before using remote tools regularly, it is a good idea to set up **SSH keys**.

This is useful because:

- you will likely use SSH directly anyway,
- it makes authentication easier,
- some remote tools can leverage SSH,
- it avoids repeated password prompts.

#### On macOS or Linux
Open a terminal on your local machine and run:

```bash
ssh-keygen -t ed25519 -C "your_email@example.com"
```

Press Enter to accept the default key location. You may set a passphrase if you like.

Then copy the public key to `newton`:

```bash
ssh-copy-id your_netid@10.246.31.148
```

If `ssh-copy-id` is not available, do it manually:

```bash
cat ~/.ssh/id_ed25519.pub
```

Copy the output, then log in to `newton` once with your password and append the key to:

```text
~/.ssh/authorized_keys
```

#### On Windows
Use either:

- **PowerShell** with OpenSSH, or
- **Git Bash**

Generate a key:

```powershell
ssh-keygen -t ed25519 -C "your_email@example.com"
```

Then either use `ssh-copy-id` if available, or manually append the contents of your `.pub` file to `~/.ssh/authorized_keys` on `newton`.

#### Test your SSH access
Once configured, test with:

```bash
ssh your_netid@10.246.31.148
```

If that works, your SSH keys are set up properly.

### 9.2: install NoMachine on your laptop or desktop

Install **NoMachine** on your local computer.

NoMachine provides a graphical remote desktop connection to `newton`, which is useful because SEISAN tools such as waveform display and picking are much easier to use in a graphical session than through a plain terminal.

#### General installation steps
1. Download NoMachine for your operating system.
2. Install it in the normal way for your platform.
3. Launch the NoMachine client.

In many campus settings, NoMachine is one of the more reliable remote desktop options.

#### Important note
Even if NoMachine does not directly use your SSH session in the way ordinary `ssh` does, you should still set up SSH keys first because:

- they are essential for terminal work,
- they simplify file transfer and command-line access,
- they give you a reliable fallback if the graphical connection has trouble.

### 9.3: connect to `newton`

Open NoMachine and create a connection to:

```text
10.246.31.148
```

Then connect.

You should be presented with a **graphical login screen**.

### 9.4: log in

Log in as **your normal user account**.

On your first login, you may see one or more NoMachine setup or welcome screens.

If so, just keep clicking:

```text
Next
```

until they are dismissed.

### 9.5: open a terminal

Once you are at the remote desktop:

1. right-click on the desktop,
2. choose **Open in Terminal**.

This gives you a bash shell on `newton`.

## 9.6: initialize SEISAN

In the terminal, type:

```bash
setup_seisan
```

This should configure the environment so that SEISAN commands are available in that shell.

After that, you should be able to run SEISAN programs such as:

```bash
eev
select
mulplt
collect
```

### 9.7: Suggested first commands on `newton`

After running `setup_seisan`, try a few simple checks:

```bash
echo $SEISAN_TOP
echo $DEF_BASE
echo $SEISAN_LOCAL
alias setup_seisan_local
which eev
which mulplt
```

These help confirm that the environment is set correctly.

## 10. Example student workflow

A typical workflow for an event analysis session might be:

1. connect to `newton` with NoMachine,
2. open a terminal,
3. run `setup_seisan`,
4. use `select` to define a subset of events,
5. browse that subset with `eev index.out`,
6. open waveforms in `mulplt`,
7. pick phases and amplitudes,
8. save the resulting readings in `REA/{DBNAME}/{YYYY}/{MM}`.

## 11. Final recommendations

If you remember only three things from this notebook, remember these:

### Continuous data
Archive them as **daily MiniSEED files **per SEED id** in SDS format**.

### Event data
Store them as **one MiniSEED file per event** in a **SEISAN-compatible `WAV` directory tree**.

### Event metadata
Store readings/results in **parallel `REA` directories** so that event interpretation stays tied to event waveform files.

This combination gives you a practical and durable workflow that works well with:

- Python/ObsPy,
- Antelope,
- and SEISAN.

In [ ]:
# This notebook is primarily explanatory.
# Add your own paths, archive locations, or course-specific examples below.

SDS_EXAMPLE = "SDS_ROOT/YYYY/NET/STA/CHAN.D/NET.STA.LOC.CHAN.D.YYYY.JJJ"
WAV_EXAMPLE = "WAV/{DBNAME}/{YYYY}/{MM}/EVENT_FILENAME"
REA_EXAMPLE = "REA/{DBNAME}/{YYYY}/{MM}/SFILE_NAME"

print("Continuous archive example:")
print(SDS_EXAMPLE)
print()
print("Event waveform archive example:")
print(WAV_EXAMPLE)
print()
print("Event metadata archive example:")
print(REA_EXAMPLE)